# 03 — Volt-VAr curtailment detection and attribution

This notebook asks a narrower question than Volt-VAr conformance: **is there empirical evidence that absorbing reactive power constrained active-power output, and how much counterfactual-supported generation may be attributed to that constraint?**

## Evidence model

- **Method A** detects an apparent-power-limit symptom and reports a reactive-headroom displacement proxy. It is not lost energy.
- **Method B** adds the GHI counterfactual and reports nested evidence tiers.
- Tier 1: high-voltage absorbing Q.
- Tier 2: absorbing Q plus operation near the selected empirical apparent-power limit.
- Tier 3: counterfactual P above the active-power headroom implied by measured Q.
- Tier 4: Tier 2 and Tier 3 together, with attributed energy limited to the part of observed loss lying above the Q-constrained headroom.

A separate required-Q scenario estimates the constraint that would arise from the standards-required Q curve. It is a scenario, not evidence of measured inverter behaviour.

In [ ]:
%matplotlib inline
import importlib
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
REPO_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'bms_sa_review').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f'Cannot find repository root from {HERE}')
LIB = REPO_ROOT / 'bms_sa_review' / 'data_query' / 'lib'
for path in (REPO_ROOT, LIB):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from bms_sa_review.shared.aws_config import aq
import analysis_contract as contract
import conformance_queries as cq
import conformance_metrics as cm
import voltvar_params as vp
import voltvar_queries as vq
import voltvar_metrics as vm
import voltvar_plots as vpl
for module in (contract, cq, cm, vp, vq, vm, vpl):
    importlib.reload(module)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 190)

In [ ]:
CONFIG = contract.AnalysisConfig(
    years=(2024, 2025), flex_selection='exclude',
    rating_basis='ac_capacity_kw', empirical_limit_basis='s_99',
    voltage_aggregation='avg', capability_profile='review_corrected',
).validate()
PARAMS = vp.VoltVarParams(
    years=CONFIG.years,
    v_low=240.0, v_high=253.0,
    peak_hour_start=11, peak_hour_end=14,
    ghi_cs_ratio_min=0.95, apply_ghi_filter=True,
    empirical_limit_basis='s_99',
    tolerance_basis='ac_capacity_kw',
    rating_basis='ac_capacity_kw',
    require_apparent_limit_symptom=True,
    flex_selection='exclude',
).validate()
display(contract.manifest(CONFIG))
display(vp.describe(PARAMS))

## 1. Input coverage and Stage 2 baseline

In [ ]:
coverage = vq.fetch_input_coverage(aq, CONFIG, PARAMS)
stage2_baseline = vq.fetch_stage2_vvar_baseline(aq, CONFIG, PARAMS)
meta = cq.fetch_metadata(aq, CONFIG)
display(coverage)
display(stage2_baseline)
display(cm.validate_metadata(meta))

## 2. Method A — empirical apparent-limit symptom

The symptom requires average site voltage in 240–253 V, absorbing Q, the selected solar/GHI window, and apparent power within the selected tolerance of the empirical limit. `S_99` is an observed limit proxy, not verified manufacturer S_rated.

In [ ]:
method_a_site_year = vq.fetch_method_a_site_year(aq, CONFIG, PARAMS)
method_a_enriched, method_a_summary = vm.method_a_summary(
    method_a_site_year, CONFIG.interval_h)
display(method_a_summary)
display(method_a_enriched.sort_values('headroom_displacement_kwh', ascending=False).head(20))
print('The headroom quantity is a symptom proxy, not a lost-energy estimate.')

## 3. Method B — counterfactual-supported attribution

Missing counterfactuals remain missing. Energy percentages use only counterfactual-covered potential generation. The measured-Q result requires an apparent-limit symptom by default; the required-Q result is displayed separately as a standards scenario.

In [ ]:
method_b_site_year = vq.fetch_method_b_site_year(aq, CONFIG, PARAMS)
method_b_enriched, method_b_summary = vm.method_b_summary(
    method_b_site_year, CONFIG.interval_h)
tiers = vm.evidence_tier_table(method_b_site_year)
display(method_b_summary)
display(tiers)
vpl.plot_evidence_tiers(tiers);

## 4. Fleet breakdowns and concentration

In [ ]:
vvar_breakdowns = {}
for group in ('state', 'dnsp', 'oem', 'install_year'):
    vvar_breakdowns[group] = vm.group_breakdown(
        method_b_enriched, meta, group, min_sites=20)
    print(f'\nVolt-VAr attribution by {group}')
    display(vvar_breakdowns[group])

vpl.plot_group_energy(vvar_breakdowns['state'], 'state',
                      'Measured-Q-attributed energy by state');

In [ ]:
ranked, concentration = vm.concentration(method_b_enriched)
display(pd.DataFrame({'top_site_share_pct': concentration.keys(),
                      'cumulative_energy_share_pct': concentration.values()}))
if not ranked.empty:
    display(ranked.head(20))
    vpl.plot_concentration(ranked, concentration);

## 5. Selected-site validation

The example is chosen from Tier 4 only. The plotted voltage is the average site voltage already stored in `structured_data_v2`, so it matches the selected production aggregation instead of selecting one circuit or substituting maximum circuit voltage.

In [ ]:
candidates = method_b_enriched[method_b_enriched['tier4_attributed_count'] > 0]
if candidates.empty:
    print('No Tier 4 site-year is available for plotting.')
else:
    chosen = candidates.sort_values('attributed_measured_q_kwh', ascending=False).iloc[0]
    EXAMPLE_SITE = int(chosen.site_id)
    EXAMPLE_YEAR = int(chosen.year)
    example_intervals = vq.fetch_method_b_intervals(
        aq, CONFIG, PARAMS, EXAMPLE_SITE, EXAMPLE_YEAR)
    display(example_intervals.head())
    vpl.plot_site_intervals(example_intervals, EXAMPLE_SITE, EXAMPLE_YEAR)
    vpl.plot_pq_circle(example_intervals, EXAMPLE_SITE);

## 6. Sensitivity analysis

These queries are intentionally opt-in because each scenario scans interval data. Run the baseline first. The minimum set varies the empirical limit, GHI ratio, and whether the apparent-limit symptom is required. `ac_capacity_kw` and `S_99` remain visibly distinct.

In [ ]:
RUN_SENSITIVITY = False
sensitivity_results = []
if RUN_SENSITIVITY:
    scenarios = {
        'baseline: S99, GHI>=0.95, symptom required': PARAMS,
        'AC capacity empirical limit': PARAMS.with_changes(empirical_limit_basis='ac_capacity_kw'),
        'GHI ratio >=0.90': PARAMS.with_changes(ghi_cs_ratio_min=0.90),
        'GHI ratio >=0.85': PARAMS.with_changes(ghi_cs_ratio_min=0.85),
        'no apparent-limit gate': PARAMS.with_changes(require_apparent_limit_symptom=False),
    }
    for label, scenario in scenarios.items():
        print('Running:', label)
        sensitivity_results.append(
            (label, vq.fetch_method_b_site_year(aq, CONFIG, scenario)))
    sensitivity_summary = vm.sensitivity_table(sensitivity_results)
    display(sensitivity_summary)
else:
    print('Set RUN_SENSITIVITY=True when ready to run the additional Athena scans.')

## 7. Reporting-ready conclusions and limitations

In [ ]:
reporting_table = pd.DataFrame([
    ['Method A symptom prevalence', 'How often did absorbing-Q operation occur near the empirical apparent-power limit?', 'Symptom only; headroom displacement is not observed lost energy'],
    ['Tier 3 counterfactual evidence', 'Was potential P above measured-Q active-power headroom?', 'Does not alone prove the constraint was binding'],
    ['Tier 4 attributed energy', 'How much observed counterfactual loss lay above a simultaneously binding measured-Q headroom?', 'Conditional on GHI model, S99 proxy, tolerance and selected voltage/time window'],
    ['Required-Q scenario', 'What active-power constraint would the standards-required Q imply?', 'Scenario, not measured behaviour'],
], columns=['reported_result','question_answered','principal_limitation'])
display(reporting_table)

assumptions = pd.DataFrame([
    ['S_99', 'Empirical apparent-power-limit proxy; not verified manufacturer S_rated'],
    ['ac_capacity_kw', 'Provider capacity used as the standards rating proxy'],
    ['Voltage', 'Average site voltage, matching the current production run'],
    ['Circuit aggregation', 'Uses site-level structured_data_v2; does not pre-filter individual circuits by voltage'],
    ['Flexible export', 'Excluded; fixed export limits and other controls may remain confounders'],
    ['Counterfactual', 'Model-dependent and quality-gated; missing values are excluded, not set to zero'],
    ['Attribution', 'Ordinary inverter controls, clipping, temperature, outages and telemetry error can confound causal interpretation'],
    ['Time conversion', 'Five-minute kW samples multiplied by 1/12 to obtain kWh'],
], columns=['assumption','treatment'])
display(assumptions)